In [1]:
%cd ../../../scaling-laws-ecnn
%load_ext autoreload
%autoreload 2

import wandb
import os
import re
import training.utils as utils
from training.one_time_tasks.wandb_add import add_block_num_layers


/home/stefan/git/scaling-laws-ecnn


ModuleNotFoundError: No module named 'torch'

In [2]:
api = wandb.Api()

filters = {
    "config.model._target_": "networks.eq_nasnet.eq_nasnet.EquivariantNASNet",
    "state": "finished",
    "tags": "constant_scheduler"
}

runs = api.runs(path=f"ga92xug/SL-Scaling", filters=filters)

for run in runs:
    print("\nCurrent name:", run.name)
    #if run.id != "5d0jx4et":
    #    continue

    try:
        utils.give_wandb_name(
            model=None,
            wandb_run=run,
            ignore_name=True,
        )
        print("New name:", run.name)
        #run.update()
    except Exception as e:
        print(e)
        print(run.name, "FAILED")

    break

NameError: name 'wandb' is not defined

In [16]:
def extract_info_from_log(run):
    file = run.file('output.log')
    file.download(replace=True)

    with open(file.name, "r") as log_file:
        console_logs = log_file.read()
        #print(console_logs)
        
        # Define a regular expression pattern to capture both epoch and learning rate
        pattern = r'Epoch (\d+): reducing learning rate of group \d+ to (\d+\.\d+e[-+]?\d+).'
        
        # Find all matches of the pattern in the content
        matches = re.findall(pattern, console_logs)
        
    os.remove("output.log")

    epochs = [int(epoch) for epoch, _ in matches]
    lrs = [float(lr) for _, lr in matches]

    return epochs, lrs

In [19]:
api = wandb.Api()

filters = {
    "config.model._target_": "networks.eq_nasnet.eq_nasnet.EquivariantNASNet",
    "config.training.epochs": 120,
    "state": "finished",

}

runs = api.runs(path=f"ga92xug/SL-Scaling", filters=filters)


info = {}
for run in runs:
    info[run.id] = {}
    epochs, lr_rates = extract_info_from_log(run)

    info[run.id]["epochs"] = epochs
    info[run.id]["lr_rates"] = lr_rates
    info[run.id]["valid_acc_weigthed"] = run.history(keys=["valid.acc_weighted"]).values[:, 1]
    info[run.id]["name"] = run.name

    break


#print(info)

{'zufkjx7q': {'epochs': [61, 72, 89, 100, 114], 'lr_rates': [0.0025, 0.00125, 0.000625, 0.0003125, 0.00015625], 'valid_acc_weigthed': array([0.25144726, 0.25835907, 0.28141743, 0.3363409 , 0.37302327,
       0.38838625, 0.46396407, 0.44244859, 0.46948016, 0.47148222,
       0.47135776, 0.49936688, 0.48025718, 0.51726335, 0.49636069,
       0.55418473, 0.55299073, 0.53719038, 0.53558779, 0.58426416,
       0.57163656, 0.5625999 , 0.59009331, 0.58295643, 0.585886  ,
       0.60647875, 0.62385386, 0.60244828, 0.6107583 , 0.59696227,
       0.57886529, 0.66188067, 0.63407654, 0.60920835, 0.59380209,
       0.63207281, 0.58912492, 0.63095534, 0.6495536 , 0.63422096,
       0.61655486, 0.67290056, 0.6432513 , 0.64208877, 0.68098086,
       0.62983841, 0.66274667, 0.65311658, 0.59612203, 0.62288296,
       0.65532559, 0.62962562, 0.64293617, 0.65821862, 0.65839434,
       0.63664544, 0.64575922, 0.65441734, 0.6684773 , 0.63780105,
       0.64255118, 0.68594491, 0.69309413, 0.6718111 , 0.67087